# DK — Language Model from Scratch

This notebook builds a decoder-only transformer language model from the ground up.
No wrapper libraries, no shortcuts. Every component is written explicitly so you actually
understand what each piece does and why it is designed that way.

The architecture follows the modern standard used by LLaMA, Mistral, and similar models:
RoPE positional encodings, RMSNorm instead of LayerNorm, SwiGLU activation in the FFN,
and pre-norm placement. These are not arbitrary choices — each one has a concrete reason,
which is explained inline.

---

**Model family:** Decoder-only (GPT-style)  
**Training objective:** Next-token prediction (causal language modeling)  
**Scale:** Configurable — default targets ~10M parameters, trainable on a laptop GPU

---

| Phase | Component | Description |
|---|---|---|
| 1 | Config | All hyperparameters in one place |
| 2 | Tokenizer | Character-level, with a BPE swap path |
| 3 | Dataset | Sliding-window next-token dataset |
| 4 | RoPE | Rotary positional embeddings |
| 5 | Attention | Multi-head causal self-attention |
| 6 | FFN | SwiGLU feed-forward network |
| 7 | Block | Full transformer block with residuals |
| 8 | Model | Complete DK language model |
| 9 | Optimizer | AdamW with proper weight decay groups |
| 10 | Training | Full loop with grad accumulation and checkpointing |
| 11 | Generation | Autoregressive sampling with temperature and top-k |

---
## Setup

Install everything before running. If you are on Colab, the CUDA build of PyTorch is
already available — just uncomment the pip install lines for the extra libraries.

In [ ]:
# Uncomment if setting up a fresh environment.
#
# !pip install torch --index-url https://download.pytorch.org/whl/cu121
# !pip install tiktoken datasets tqdm matplotlib

import math
import os
import time
from dataclasses import dataclass
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


# Pick the best available device automatically.
# MPS is Apple Silicon — bfloat16 support is incomplete there, so
# we will fall back to float32. CPU is the last resort.
device = (
    "cuda" if torch.cuda.is_available() else
    "mps"  if torch.backends.mps.is_available() else
    "cpu"
)

print(f"Device  : {device}")

if device == "cuda":
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU     : {name}")
    print(f"VRAM    : {vram:.1f} GB")

print(f"PyTorch : {torch.__version__}")

---
## Phase 1 — Configuration

All model and training hyperparameters live in one dataclass. If you want to experiment,
change a value here and it propagates everywhere automatically.

The default config (`d_model=384, n_layers=6, n_heads=6`) gives you roughly 10 million
parameters — small enough to train on a laptop GPU in under an hour on a decent dataset,
large enough to produce coherent text.

In [ ]:
@dataclass
class DKConfig:

    # ---------------------------------------------------------------
    # Model architecture
    # ---------------------------------------------------------------

    # Set dynamically after building the tokenizer.
    # Switch to tiktoken GPT-2 BPE and set this to 50257.
    vocab_size: int = 65

    # How many tokens the model sees at once. Longer context means
    # better coherence over longer text, but attention cost grows
    # quadratically. 256 is fine for a first experiment.
    context_len: int = 256

    # The core embedding dimension. Every token is represented as a
    # d_model-dimensional vector throughout the whole network.
    d_model: int = 384

    # Number of attention heads. Must divide d_model evenly.
    n_heads: int = 6

    # Number of transformer blocks stacked on top of each other.
    n_layers: int = 6

    # FFN hidden dimension multiplier. Standard is 4x.
    # With SwiGLU we use roughly 2.67x to keep param count similar.
    ffn_mult: int = 4

    # Set to 0.0 for large-data regimes — dropout helps most when
    # you are capacity-limited, not data-limited.
    dropout: float = 0.1

    # Modern models (LLaMA, Mistral) remove biases entirely.
    # They add parameters without meaningfully improving loss.
    bias: bool = False

    # ---------------------------------------------------------------
    # Training
    # ---------------------------------------------------------------

    batch_size: int = 64

    # Gradient accumulation: effective batch = batch_size x grad_accum.
    # Use this to simulate large batches when VRAM is limited.
    grad_accum: int = 4

    max_iters: int = 5000
    eval_interval: int = 500
    eval_iters: int = 200
    log_interval: int = 100

    # ---------------------------------------------------------------
    # Optimizer
    # ---------------------------------------------------------------

    # 3e-4 is the classic starting point for small models.
    # Scale down to 1e-4 or 6e-5 as you go larger.
    lr: float = 3e-4

    weight_decay: float = 0.1
    beta1: float = 0.9
    beta2: float = 0.95    # lower than the default 0.999 — standard for LLM training
    grad_clip: float = 1.0
    warmup_iters: int = 200

    # ---------------------------------------------------------------
    # Paths
    # ---------------------------------------------------------------
    data_path: str = "data/input.txt"
    ckpt_path: str = "dk_checkpoint.pt"


cfg = DKConfig()

assert cfg.d_model % cfg.n_heads == 0, (
    f"d_model ({cfg.d_model}) must be divisible by n_heads ({cfg.n_heads})"
)
cfg.head_dim = cfg.d_model // cfg.n_heads

# Rough parameter estimate — actual count comes from model.num_params().
est = 12 * cfg.n_layers * cfg.d_model ** 2

print("DK config loaded")
print(f"  Architecture    : {cfg.n_layers}L x {cfg.n_heads}H x {cfg.d_model}d")
print(f"  Context length  : {cfg.context_len} tokens")
print(f"  Estimated params: ~{est / 1e6:.1f}M")
print(f"  Effective batch : {cfg.batch_size * cfg.grad_accum}")

---
## Phase 2 — Tokenizer

A tokenizer maps raw text to integer token IDs and back. We start with a character-level
tokenizer — every unique character in the corpus becomes one token. The vocabulary is tiny
(60-100 chars for English), which makes the embedding table small and the output
distribution easy to learn. The downside is that the model processes text character by
character, which is less efficient than BPE.

If you want to switch to GPT-2 tokenizer (50k-token BPE), uncomment the tiktoken block
below. The rest of the notebook needs zero changes.

In [ ]:
class CharTokenizer:
    """
    Character-level tokenizer.

    Builds its vocabulary from whatever text you pass at construction time.
    Every unique character becomes one token ID. Deterministic: same text
    always produces the same vocabulary in the same order.
    """

    def __init__(self, text: str):
        chars = sorted(set(text))
        self.vocab_size = len(chars)
        self.stoi = {ch: i for i, ch in enumerate(chars)}
        self.itos = {i: ch for ch, i in self.stoi.items()}

    def encode(self, text: str) -> list[int]:
        return [self.stoi[ch] for ch in text]

    def decode(self, ids: list[int]) -> str:
        return "".join(self.itos[i] for i in ids)

    def __repr__(self):
        return f"CharTokenizer(vocab_size={self.vocab_size})"


# --- Optional: drop in tiktoken for GPT-2 BPE ----------------------------
#
# import tiktoken
# enc = tiktoken.get_encoding("gpt2")
#
# class TiktokenWrapper:
#     vocab_size = enc.n_vocab  # 50257
#     def encode(self, text): return enc.encode(text)
#     def decode(self, ids):  return enc.decode(ids)
#
# -------------------------------------------------------------------------

# Round-trip test.
sample = "DK Language Model — trained from scratch."
demo = CharTokenizer(sample)
ids = demo.encode(sample)
assert demo.decode(ids) == sample, "Encode/decode round-trip failed."

print(demo)
print(f"Sample  : {sample}")
print(f"Encoded : {ids[:14]} ...")
print(f"Round-trip OK")

### Load data

Tiny Shakespeare is the classic starting corpus — 1MB of text, quick to iterate on.
For more serious training, FineWeb or OpenWebText from HuggingFace are good choices.

In [ ]:
# Download Tiny Shakespeare (run once):
#
# !mkdir -p data
# !wget -q -O data/input.txt \
#   https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
#
# Then replace the synthetic text below with:
#   with open(cfg.data_path, 'r') as f:
#       text = f.read()

# Synthetic corpus for running the notebook end-to-end without a download.
text = (
    "DK is a language model trained from scratch. "
    "It learns to predict the next token given all previous tokens. "
    "The transformer architecture allows it to attend to any position in context. "
    "Attention, feed-forward layers, and residual connections work together. "
) * 600

# Build the tokenizer on the full corpus.
tokenizer = CharTokenizer(text)
cfg.vocab_size = tokenizer.vocab_size

# Tokenize and store as a flat tensor.
data = torch.tensor(tokenizer.encode(text), dtype=torch.long)

# Standard 90/10 split.
n_train = int(0.9 * len(data))
train_data = data[:n_train]
val_data   = data[n_train:]

print(f"Corpus       : {len(text):,} chars")
print(f"Vocabulary   : {cfg.vocab_size} unique tokens")
print(f"Train tokens : {len(train_data):,}")
print(f"Val tokens   : {len(val_data):,}")

---
## Phase 3 — Dataset and DataLoader

Language model training works on sliding windows over the token sequence.
Each sample is a window of `context_len` tokens as input, and the same window
shifted one position to the right as the target — because the model's job at
each position is to predict what comes next.

This gives us `context_len` gradient signals per sample, not just one, which
makes training much more data-efficient.

In [ ]:
class DKDataset(Dataset):
    """
    Sliding-window next-token prediction dataset.

    For a sequence [t0, t1, t2, t3, t4] with context_len=3:
        x = [t0, t1, t2]   (input)
        y = [t1, t2, t3]   (target, shifted by 1)

    The model is trained to predict y[i] given x[:i+1] at every
    position simultaneously during a single forward pass.
    """

    def __init__(self, data: torch.Tensor, context_len: int):
        self.data = data
        self.ctx = context_len

    def __len__(self):
        return len(self.data) - self.ctx

    def __getitem__(self, idx: int):
        chunk = self.data[idx : idx + self.ctx + 1]
        return chunk[:-1], chunk[1:]


train_ds = DKDataset(train_data, cfg.context_len)
val_ds   = DKDataset(val_data,   cfg.context_len)

train_loader = DataLoader(
    train_ds, batch_size=cfg.batch_size,
    shuffle=True, pin_memory=(device == "cuda"),
    drop_last=True, num_workers=0,
)
val_loader = DataLoader(
    val_ds, batch_size=cfg.batch_size,
    shuffle=False, pin_memory=(device == "cuda"),
    drop_last=True,
)

x_s, y_s = next(iter(train_loader))
print(f"Batch x : {x_s.shape}  (batch_size, context_len)")
print(f"Batch y : {y_s.shape}  (same shape, shifted by 1)")

---
## Phase 4 — Rotary Position Embeddings (RoPE)

The original transformer added a fixed sinusoidal table to token embeddings before the
first layer. RoPE does something more elegant: it encodes position directly into the
query and key vectors inside each attention head by rotating them.

Why this matters: the dot product between a rotated Q and a rotated K naturally
depends only on the *relative* distance between positions, not absolute positions.
The model generalizes better to lengths it has not seen during training, and you
do not need to add anything to the embedding.

We precompute the cosine and sine frequency tables once, register them as buffers
so they move to GPU with the model, and slice them per sequence length at runtime.

In [ ]:
def precompute_rope_freqs(
    head_dim: int,
    max_len: int,
    theta: float = 10_000.0,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Precompute RoPE cosine and sine frequency tables.

    The frequency for dimension pair i is:
        theta_i = 1 / (theta ^ (2i / head_dim))

    Returns two tensors of shape (max_len, head_dim // 2).
    """
    freqs = 1.0 / (
        theta ** (torch.arange(0, head_dim, 2).float() / head_dim)
    )
    positions = torch.arange(max_len)
    table = torch.outer(positions, freqs)  # (max_len, head_dim // 2)
    return table.cos(), table.sin()


def apply_rope(
    q: torch.Tensor,
    k: torch.Tensor,
    cos: torch.Tensor,
    sin: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Apply rotary embeddings to queries and keys.

    Each pair of adjacent dimensions (x1, x2) is rotated by angle theta_i * pos:
        x1' = x1 * cos - x2 * sin
        x2' = x1 * sin + x2 * cos

    q, k : (B, n_heads, T, head_dim)
    cos  : (T, head_dim // 2)
    sin  : (T, head_dim // 2)
    """

    def rotate(x: torch.Tensor) -> torch.Tensor:
        x1, x2 = x[..., ::2], x[..., 1::2]
        T = x.shape[2]
        c = cos[:T].unsqueeze(0).unsqueeze(0)
        s = sin[:T].unsqueeze(0).unsqueeze(0)
        return torch.stack([x1 * c - x2 * s, x1 * s + x2 * c], dim=-1).flatten(-2)

    return rotate(q), rotate(k)


cos_cached, sin_cached = precompute_rope_freqs(cfg.head_dim, cfg.context_len)
cos_cached = cos_cached.to(device)
sin_cached = sin_cached.to(device)

print(f"RoPE tables: cos {cos_cached.shape}, sin {sin_cached.shape}")
print(f"(positions={cfg.context_len}, freq pairs={cfg.head_dim // 2})")

---
## Phase 5 — Causal Self-Attention

Self-attention lets every token attend to every other token in the sequence.
Causal means the upper triangle of the attention matrix is masked — a token at
position i can only see positions 0 through i. This is what makes the model
autoregressive: it cannot peek at future tokens during training.

We use `F.scaled_dot_product_attention` from PyTorch 2.0. This is not just a
convenience wrapper — when your GPU supports it, PyTorch dispatches to a
FlashAttention kernel automatically, which avoids materializing the full
attention matrix in memory. The memory saving is significant at longer contexts.

In [ ]:
class CausalSelfAttention(nn.Module):
    """
    Multi-head causal self-attention with RoPE position encoding.

    Q, K, V projections are fused into a single linear for efficiency.
    V is not rotated — it carries content, not position.
    """

    def __init__(self, cfg: DKConfig):
        super().__init__()
        self.n_heads  = cfg.n_heads
        self.head_dim = cfg.head_dim
        self.d_model  = cfg.d_model

        # One linear instead of three separate Q/K/V projections.
        self.qkv_proj  = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=cfg.bias)
        self.out_proj  = nn.Linear(cfg.d_model, cfg.d_model, bias=cfg.bias)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.attn_drop  = cfg.dropout

    def forward(
        self,
        x: torch.Tensor,
        cos: torch.Tensor,
        sin: torch.Tensor,
    ) -> torch.Tensor:
        B, T, C = x.shape
        H, D = self.n_heads, self.head_dim

        # Project and split into Q, K, V.
        q, k, v = self.qkv_proj(x).split(self.d_model, dim=-1)

        # Reshape to per-head tensors: (B, H, T, D).
        q = q.view(B, T, H, D).transpose(1, 2)
        k = k.view(B, T, H, D).transpose(1, 2)
        v = v.view(B, T, H, D).transpose(1, 2)

        # Apply RoPE to Q and K.
        q, k = apply_rope(q, k, cos, sin)

        # Scaled dot-product attention.
        # is_causal=True generates the causal mask internally,
        # which is more efficient than passing an explicit mask tensor.
        y = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.attn_drop if self.training else 0.0,
            is_causal=True,
        )

        # Merge heads and project to d_model.
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.out_proj(y))


# Shape check.
_a = CausalSelfAttention(cfg).to(device)
_x = torch.randn(2, cfg.context_len, cfg.d_model, device=device)
_o = _a(_x, cos_cached, sin_cached)
print(f"Attention input  : {_x.shape}")
print(f"Attention output : {_o.shape}  (should match)")
del _a, _x, _o

---
## Phase 6 — Feed-Forward Network (SwiGLU)

After attention mixes information across positions, the FFN applies a non-linear
transformation at each position independently. Think of attention as routing
information and FFN as processing it.

We use SwiGLU instead of the original ReLU FFN. SwiGLU adds a gating mechanism —
one branch gates another — giving the model more expressive power at the same
parameter count. It is the activation used in LLaMA, PaLM, and Gemma.

The hidden dimension is set to 2/3 of the standard 4x to keep total parameter
count comparable to a vanilla FFN, since SwiGLU needs an extra projection matrix.

In [ ]:
class SwiGLU(nn.Module):
    """
    SwiGLU feed-forward network.

    Standard FFN : FFN(x) = ReLU(xW1) W2
    SwiGLU       : FFN(x) = (SiLU(xW_gate) * xW_up) W_down

    The element-wise product between the gated branch and the up-projected
    branch lets the network suppress or amplify each hidden dimension
    independently based on a learned gating signal.
    """

    def __init__(self, cfg: DKConfig):
        super().__init__()

        # 2/3 of 4x rounded up to the nearest 64 for memory alignment.
        hidden = int(cfg.d_model * cfg.ffn_mult * 2 / 3)
        hidden = 64 * math.ceil(hidden / 64)

        self.gate = nn.Linear(cfg.d_model, hidden, bias=cfg.bias)
        self.up   = nn.Linear(cfg.d_model, hidden, bias=cfg.bias)
        self.down = nn.Linear(hidden, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))


_f = SwiGLU(cfg).to(device)
_x = torch.randn(2, cfg.context_len, cfg.d_model, device=device)
print(f"FFN input  : {_x.shape}")
print(f"FFN output : {_f(_x).shape}  (should match)")
del _f, _x

---
## Phase 7 — Transformer Block

One transformer block is attention followed by FFN, each wrapped in a residual
connection and a normalization layer.

We use **pre-norm** — normalization happens before each sublayer. The original paper
used post-norm, but pre-norm is significantly more stable at scale and has been
standard since GPT-2.

We use **RMSNorm** instead of LayerNorm. LayerNorm computes both mean and variance.
RMSNorm only computes the root-mean-square, skipping the mean subtraction. Empirically
identical quality, ~10% faster. LLaMA, Mistral, and Gemma all use it.

In [ ]:
class DKBlock(nn.Module):
    """
    One DK transformer block.

        x = x + Attention(RMSNorm(x))   <- pre-norm, residual
        x = x + FFN(RMSNorm(x))         <- pre-norm, residual

    The residual connection is what makes deep transformers trainable.
    Without it, gradients from the output layer cannot reach early layers.
    """

    def __init__(self, cfg: DKConfig):
        super().__init__()
        self.norm1 = nn.RMSNorm(cfg.d_model)
        self.attn  = CausalSelfAttention(cfg)
        self.norm2 = nn.RMSNorm(cfg.d_model)
        self.ffn   = SwiGLU(cfg)

    def forward(
        self,
        x: torch.Tensor,
        cos: torch.Tensor,
        sin: torch.Tensor,
    ) -> torch.Tensor:
        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.ffn(self.norm2(x))
        return x


_b = DKBlock(cfg).to(device)
_x = torch.randn(2, cfg.context_len, cfg.d_model, device=device)
print(f"Block input  : {_x.shape}")
print(f"Block output : {_b(_x, cos_cached, sin_cached).shape}")
del _b, _x

---
## Phase 8 — DK Language Model

The full model connects everything:

```
input tokens
    -> token embedding     (vocab_size -> d_model)
    -> N transformer blocks
    -> final RMSNorm
    -> LM head             (d_model -> vocab_size)
    -> logits over vocabulary
```

**Weight tying:** The LM head and token embedding share the same weight matrix.
Both are doing the same thing in opposite directions — embedding maps a token ID
to a dense vector, LM head maps a dense vector back to token scores. Tying them
halves the parameter count for that matrix and consistently improves perplexity.
All major LLMs use this.

In [ ]:
class DKLanguageModel(nn.Module):
    """
    DK — decoder-only language model.

    During training, targets are passed in and loss is returned.
    During inference, only the last position's logits are computed
    to avoid redundant work on the full sequence.
    """

    def __init__(self, cfg: DKConfig):
        super().__init__()
        self.cfg = cfg

        # RoPE handles position, so no positional embedding table.
        self.tok_emb  = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.emb_drop = nn.Dropout(cfg.dropout)

        self.blocks = nn.ModuleList(
            [DKBlock(cfg) for _ in range(cfg.n_layers)]
        )

        self.norm_f  = nn.RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        # Tie weights.
        self.lm_head.weight = self.tok_emb.weight

        # Register RoPE tables as buffers — they move to GPU with the model.
        cos, sin = precompute_rope_freqs(cfg.head_dim, cfg.context_len)
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)

        self.apply(self._init_weights)

        # Scale residual output projections by 1/sqrt(2 * n_layers).
        # This keeps the residual stream well-scaled at initialization
        # regardless of depth. GPT-2 introduced this, still used today.
        scale = 0.02 / math.sqrt(2 * cfg.n_layers)
        for name, p in self.named_parameters():
            if name.endswith(("out_proj.weight", "down.weight")):
                nn.init.normal_(p, mean=0.0, std=scale)

    def _init_weights(self, module: nn.Module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(
        self,
        idx: torch.Tensor,
        targets: Optional[torch.Tensor] = None,
    ) -> tuple[torch.Tensor, Optional[torch.Tensor]]:
        B, T = idx.shape
        assert T <= self.cfg.context_len, (
            f"Input length {T} exceeds context_len {self.cfg.context_len}."
        )

        x = self.emb_drop(self.tok_emb(idx))   # (B, T, d_model)

        # Slice RoPE tables to the current sequence length.
        cos = self.cos[:T]
        sin = self.sin[:T]

        for block in self.blocks:
            x = block(x, cos, sin)

        x = self.norm_f(x)

        if targets is not None:
            # Training: compute logits and loss for every position.
            logits = self.lm_head(x)
            loss = F.cross_entropy(
                logits.view(-1, self.cfg.vocab_size),
                targets.view(-1),
            )
        else:
            # Inference: only last position needed.
            logits = self.lm_head(x[:, [-1], :])
            loss = None

        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        idx: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: Optional[int] = None,
    ) -> torch.Tensor:
        """
        Autoregressively sample tokens one at a time.

        temperature < 1.0 : more focused, less varied
        temperature > 1.0 : more random, more creative
        top_k             : only sample from the top-k most probable tokens
        """
        self.eval()

        for _ in range(max_new_tokens):
            idx_window = idx[:, -self.cfg.context_len :]

            logits, _ = self(idx_window)
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                top_values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits = logits.masked_fill(logits < top_values[:, [-1]], float("-inf"))

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)

        return idx

    def num_params(self, exclude_embeddings: bool = True) -> int:
        n = sum(p.numel() for p in self.parameters())
        if exclude_embeddings:
            n -= self.tok_emb.weight.numel()
        return n


# Build the model.
model = DKLanguageModel(cfg).to(device)

total = sum(p.numel() for p in model.parameters())
print("DK Language Model")
print(f"  Parameters (total)       : {total / 1e6:.3f}M")
print(f"  Parameters (ex. embed)   : {model.num_params() / 1e6:.3f}M")
print(f"  Layers / Heads / d_model : {cfg.n_layers} / {cfg.n_heads} / {cfg.d_model}")
print(f"  Context length           : {cfg.context_len} tokens")

# Verify the forward pass works and the initial loss is near log(vocab_size).
_i = torch.randint(0, cfg.vocab_size, (2, cfg.context_len), device=device)
_t = torch.randint(0, cfg.vocab_size, (2, cfg.context_len), device=device)
_logits, _loss = model(_i, _t)

print(f"\nForward pass  : OK")
print(f"Output shape  : {_logits.shape}")
print(f"Initial loss  : {_loss.item():.4f}  (expected near {math.log(cfg.vocab_size):.2f})")
del _i, _t, _logits, _loss

---
## Phase 9 — Optimizer and Learning Rate Schedule

We use **AdamW** — the standard for transformer training. Regular Adam applies weight
decay incorrectly through the gradient update. AdamW applies it directly to the
weights as a separate step. This distinction matters noticeably at scale.

Weight decay is applied only to 2D weight matrices, never to biases, normalization
scale factors, or embedding tables. Applying decay to those rarely helps and can hurt.

The learning rate follows a **cosine schedule with linear warmup**. Warmup prevents
instability in the first few hundred steps when gradient estimates are noisy. Cosine
decay then smoothly reduces the learning rate, letting the model settle without
oscillating.

In [ ]:
def configure_optimizer(model: nn.Module, cfg: DKConfig) -> torch.optim.AdamW:
    """
    AdamW with separate parameter groups for weight decay.

    2D params (weight matrices) get weight decay.
    1D params (biases, RMSNorm scale) do not.
    """
    decay, no_decay = [], []

    for _, param in model.named_parameters():
        if not param.requires_grad:
            continue
        (decay if param.dim() >= 2 else no_decay).append(param)

    groups = [
        {"params": decay,    "weight_decay": cfg.weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]

    # Fused AdamW runs a faster CUDA kernel when available.
    use_fused = device == "cuda" and "fused" in (
        torch.optim.AdamW.__init__.__code__.co_varnames
    )

    optimizer = torch.optim.AdamW(
        groups, lr=cfg.lr, betas=(cfg.beta1, cfg.beta2), fused=use_fused
    )

    n_d  = sum(p.numel() for p in decay)
    n_nd = sum(p.numel() for p in no_decay)
    print(f"AdamW  (fused={use_fused})")
    print(f"  Weight-decayed : {n_d:,}")
    print(f"  No-decay       : {n_nd:,}")
    return optimizer


def get_lr(step: int, cfg: DKConfig) -> float:
    """
    Cosine LR schedule with linear warmup.

    0 -> warmup_iters    : linear ramp from 0 to lr
    warmup -> max_iters  : cosine decay to 0.1 * lr
    beyond max_iters     : fixed at 0.1 * lr
    """
    min_lr = cfg.lr * 0.1

    if step < cfg.warmup_iters:
        return cfg.lr * step / cfg.warmup_iters
    if step > cfg.max_iters:
        return min_lr

    progress = (step - cfg.warmup_iters) / (cfg.max_iters - cfg.warmup_iters)
    return min_lr + 0.5 * (1.0 + math.cos(math.pi * progress)) * (cfg.lr - min_lr)


# Visualise the schedule.
steps = list(range(cfg.max_iters + 1))
lrs   = [get_lr(s, cfg) for s in steps]

plt.figure(figsize=(9, 3))
plt.plot(steps, lrs, color="#3a7dc9", linewidth=2)
plt.axvline(
    cfg.warmup_iters, color="#e07b3a", linestyle="--", alpha=0.8,
    label=f"warmup end ({cfg.warmup_iters} steps)"
)
plt.xlabel("Step")
plt.ylabel("Learning rate")
plt.title("DK — cosine LR schedule with warmup")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

optimizer = configure_optimizer(model, cfg)

---
## Phase 10 — Training Loop

A few things to understand before reading the loop:

**Gradient accumulation** runs multiple micro-batches before calling `optimizer.step()`.
Each adds to the accumulated gradient. The loss is divided by `grad_accum` so the
total gradient magnitude is identical to what you would get from a single large batch.

**Mixed precision** trains in bfloat16 on CUDA, halving memory usage and speeding up
the forward pass on Ampere GPUs and newer. `GradScaler` handles loss scaling to
prevent underflow — though bfloat16 rarely needs it, it is included for compatibility.

**Gradient clipping** clips the global gradient norm to `grad_clip`. This prevents
large gradient spikes — common early in training — from destabilizing the weights.

In [ ]:
@torch.no_grad()
def estimate_loss(
    model: nn.Module,
    loaders: dict,
    eval_iters: int,
) -> dict[str, float]:
    """
    Average loss over eval_iters batches for each split.

    A single batch is too noisy to be a meaningful metric — a lucky draw
    can shift the apparent loss by 0.1 nats or more. Averaging over
    200 batches gives a much more reliable estimate.
    """
    model.eval()
    out = {}

    for split, loader in loaders.items():
        losses = []
        for i, (x, y) in enumerate(loader):
            if i >= eval_iters:
                break
            x, y = x.to(device), y.to(device)
            _, loss = model(x, y)
            losses.append(loss.item())
        out[split] = float(np.mean(losses))

    model.train()
    return out

In [ ]:
history     = {"step": [], "train_loss": [], "val_loss": [], "lr": []}
best_val    = float("inf")
loaders     = {"train": train_loader, "val": val_loader}
train_iter  = iter(train_loader)
scaler      = torch.amp.GradScaler(enabled=(device == "cuda"))

model.train()

print("Starting DK training")
print(f"  Steps          : {cfg.max_iters}")
print(f"  Effective batch: {cfg.batch_size * cfg.grad_accum}")
print(f"  Eval every     : {cfg.eval_interval} steps")
print("-" * 55)

t0 = time.perf_counter()

for step in range(1, cfg.max_iters + 1):

    # Update LR for this step.
    lr = get_lr(step, cfg)
    for g in optimizer.param_groups:
        g["lr"] = lr

    # Evaluation.
    if step % cfg.eval_interval == 0 or step == cfg.max_iters:
        losses  = estimate_loss(model, loaders, cfg.eval_iters)
        elapsed = time.perf_counter() - t0

        print(
            f"step {step:5d}  "
            f"train {losses['train']:.4f}  "
            f"val {losses['val']:.4f}  "
            f"lr {lr:.2e}  "
            f"{elapsed:.1f}s"
        )

        history["step"].append(step)
        history["train_loss"].append(losses["train"])
        history["val_loss"].append(losses["val"])
        history["lr"].append(lr)

        if losses["val"] < best_val:
            best_val = losses["val"]
            torch.save(
                {
                    "step": step,
                    "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "cfg": cfg,
                    "val_loss": best_val,
                },
                cfg.ckpt_path,
            )
            print(f"  checkpoint saved  (val_loss={best_val:.4f})")

        t0 = time.perf_counter()
        model.train()

    # Gradient accumulation loop.
    optimizer.zero_grad(set_to_none=True)

    for _ in range(cfg.grad_accum):
        try:
            x, y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            x, y = next(train_iter)

        x, y = x.to(device), y.to(device)

        amp_dtype  = torch.bfloat16 if device == "cuda" else torch.float32
        amp_device = "cuda" if device == "cuda" else "cpu"

        with torch.amp.autocast(device_type=amp_device, dtype=amp_dtype):
            _, loss = model(x, y)
            loss = loss / cfg.grad_accum   # normalise so total gradient ~ full batch

        scaler.scale(loss).backward()

    # Unscale before clipping so the clip threshold is in real gradient units.
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)

    scaler.step(optimizer)
    scaler.update()

    if step % cfg.log_interval == 0:
        print(f"  step {step:5d}  lr {lr:.2e}")


print("\nTraining complete.")
print(f"Best val loss   : {best_val:.4f}")
print(f"Best perplexity : {math.exp(best_val):.2f}")

---
## Training Curves

Loss should decrease steadily. Validation loss trailing well above training loss
is a sign of overfitting — try increasing dropout or getting more data.

The perplexity plot gives you a more intuitive sense of progress: a perplexity of N
means the model is roughly as confused as if it were choosing uniformly from N options
at each step. Smaller is better.

In [ ]:
if not history["step"]:
    print("No training history yet. Run the training loop first.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    ax = axes[0]
    ax.plot(history["step"], history["train_loss"], label="train", color="#3a7dc9", linewidth=2)
    ax.plot(history["step"], history["val_loss"],   label="val",   color="#c94a3a", linewidth=2)
    ax.set_xlabel("Step")
    ax.set_ylabel("Cross-entropy loss")
    ax.set_title("DK — loss")
    ax.legend()
    ax.grid(alpha=0.25)

    ax = axes[1]
    val_ppl = [math.exp(l) for l in history["val_loss"]]
    ax.plot(history["step"], val_ppl, color="#3aa87d", linewidth=2)
    ax.set_xlabel("Step")
    ax.set_ylabel("Perplexity")
    ax.set_title("DK — validation perplexity")
    ax.grid(alpha=0.25)

    plt.tight_layout()
    plt.savefig("dk_training_curves.png", dpi=150)
    plt.show()

    print(f"Final val loss  : {history['val_loss'][-1]:.4f}")
    print(f"Final perplexity: {val_ppl[-1]:.2f}")

---
## Phase 11 — Text Generation

Autoregressive generation asks the model the same question repeatedly: given everything
so far, what token comes next? Sample from that distribution, append the result, and
repeat. Each step sees its own output as new context.

Temperature and top-k let you trade coherence for creativity:

- **temperature 0.5** — safe and repetitive. Picks the most likely tokens consistently.
- **temperature 0.8** — a reasonable default. Some variety without going off the rails.
- **temperature 1.2** — inventive, occasionally incoherent. Good for creative tasks.
- **top_k 40** — never samples from tokens outside the 40 most probable. Prevents
  the model from occasionally outputting something completely implausible.

In [ ]:
def generate_text(
    model: nn.Module,
    tokenizer: CharTokenizer,
    prompt: str,
    max_new_tokens: int = 200,
    temperature: float = 0.8,
    top_k: int = 40,
) -> str:
    """Encode a prompt, run the model, decode the output."""
    ids = tokenizer.encode(prompt)
    idx = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    with torch.no_grad():
        out = model.generate(
            idx,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
        )

    return tokenizer.decode(out[0].tolist())


prompt = "DK"

print(f'Prompt: "{prompt}"')
print("=" * 55)

for temp in [0.5, 0.8, 1.2]:
    result = generate_text(
        model, tokenizer, prompt,
        max_new_tokens=150,
        temperature=temp,
        top_k=40,
    )
    print(f"\ntemperature={temp}")
    print("-" * 40)
    print(result)

---
## Checkpoint Utilities

In [ ]:
def save_checkpoint(model: nn.Module, optimizer, step: int, val_loss: float, path: str):
    torch.save(
        {
            "step": step,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "cfg": model.cfg,
            "val_loss": val_loss,
        },
        path,
    )
    print(f"Saved {path}")


def load_checkpoint(path: str, device: str):
    """
    Load a DK checkpoint. Returns (model, cfg).
    Pass the cfg to configure_optimizer() to resume training.
    """
    ckpt = torch.load(path, map_location=device)
    cfg  = ckpt["cfg"]
    m    = DKLanguageModel(cfg).to(device)
    m.load_state_dict(ckpt["model_state"])
    print(f"Loaded step {ckpt['step']}  val_loss={ckpt['val_loss']:.4f}")
    return m, cfg


# Usage:
# save_checkpoint(model, optimizer, step=5000, val_loss=2.10, path="dk_final.pt")
# model, cfg = load_checkpoint("dk_final.pt", device)

print("Checkpoint utilities ready.")

---
## What to Do Next

The model is complete and functional. Here are the natural next steps, roughly
in order of impact:

**Get real data.** TinyStories (1GB of simple English stories) is the best first corpus
— small enough to train quickly, structured enough that quality is easy to judge.
OpenWebText and FineWeb are good for general language modeling. Both are available via
HuggingFace `datasets`.

**Scale the architecture.** Increase `d_model`, `n_layers`, and `n_heads` proportionally.
A 125M-parameter model (`d_model=768, n_layers=12, n_heads=12`) is GPT-2 small.
Expect to need a multi-GPU machine or a long training run for anything that size.

**torch.compile.** Add `model = torch.compile(model)` after instantiation. PyTorch
traces and compiles the graph, giving 20-30% throughput improvement on modern hardware
with no code changes beyond that one line.

**Multi-GPU training.** Wrap the model in `DistributedDataParallel` and launch with
`torchrun --nproc_per_node=N`. DDP handles gradient synchronization across GPUs.

**Fine-tuning.** Once you have a pretrained base, use LoRA from the `peft` library
to fine-tune on instruction-following data. LoRA trains small low-rank adapter matrices
while freezing the base weights, which is far more efficient than full fine-tuning.

---
*Project DK — language model built from scratch.*